# HapticCodec PCA Baseline Colab Workflow

This notebook restores the prepared dataset from Google Drive, rebuilds the fixed subset, reuses an existing codec checkpoint, and runs the **post-hoc PCA baseline**:
- extract pooled codec features
- fit PCA and a linear control-to-sequence regressor
- build controls / gallery / tables
- run extended validation

No control-branch training is performed.

In [ ]:
# Experiment configuration
REPO_URL = "https://github.com/cindy-77jiayi/thesis_hapticAE.git"
BRANCH = "codex/wavcaps-vae-cleanup"
REPO_DIR = "/content/thesis_hapticAE"

RUN_NAME = "hapticcodec_pca_baseline"
OUTPUT_ROOT = "/content/outputs"
OUTPUT_RUN_DIR = f"{OUTPUT_ROOT}/{RUN_NAME}"
ARTIFACTS_DIR = "/content/drive/MyDrive/thesis_wavcaps_as/experiments/hapticcodec_subset_2000"
CODEC_CHECKPOINT_DRIVE = f"{ARTIFACTS_DIR}/hapticcodec_subset_2000/codec/best_model.pt"
CODEC_CHECKPOINT_LOCAL = "/content/best_model_codec.pt"

PREPARED_ARCHIVE_DRIVE = "/content/drive/MyDrive/thesis_wavcaps_as/prepared_dataset/wavcaps_haptic_prepared_as.tar"
PREPARED_RESTORE_ROOT = "/content/data"
PREPARED_DATA_DIR = f"{PREPARED_RESTORE_ROOT}/wavcaps_haptic_prepared_as"
SUBSET_ROOT = f"/content/data/subsets/{RUN_NAME}"
SUBSET_DATA_DIR = f"{SUBSET_ROOT}/data"
TRAIN_LIST_PATH = f"{SUBSET_ROOT}/train_list.txt"
VAL_LIST_PATH = f"{SUBSET_ROOT}/val_list.txt"
SUBSET_MANIFEST_PATH = f"{SUBSET_ROOT}/subset_manifest.jsonl"
RUNTIME_CONFIG_DIR = "/content/runtime_configs"
BASELINE_CONFIG_PATH = f"{RUNTIME_CONFIG_DIR}/{RUN_NAME}.yaml"
EXPORT_DIR = f"/content/drive/MyDrive/thesis_wavcaps_as/experiments/{RUN_NAME}"

SUBSET_SIZE = 2000
SUBSET_SEED = 42
TRAIN_SPLIT = 0.8

SR = 8000
T = 40000
ANALYSIS_BATCH_SIZE = 8
N_COMPONENTS = 8
RIDGE_ALPHA = 1.0

CHANNELS = [32, 64, 128, 128, 64]
STRIDES = [5, 4, 4, 2, 2]
FIRST_KERNEL = 25
KERNEL_SIZE = 9
RESIDUAL_KERNEL_SIZE = 7
CODE_DIM = 64
N_CODEBOOKS = 4
CODEBOOK_SIZE = 256
COMMITMENT_WEIGHT = 0.25
CONTROL_DIM = 16
CONTROL_HIDDEN = 128

In [ ]:
# Clone repo and install deps
!git clone "$REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!git checkout "$BRANCH"
!pip install -q -r requirements.txt pyyaml soundfile librosa scikit-learn tqdm matplotlib

In [ ]:
# Mount Google Drive, restore prepared dataset, and copy codec checkpoint locally
from google.colab import drive
from pathlib import Path
import shutil
import subprocess

drive.mount("/content/drive")
Path(PREPARED_RESTORE_ROOT).mkdir(parents=True, exist_ok=True)
assert Path(PREPARED_ARCHIVE_DRIVE).exists(), f"Missing archive: {PREPARED_ARCHIVE_DRIVE}"
if not Path(PREPARED_DATA_DIR).exists():
    subprocess.run(["tar", "-xf", PREPARED_ARCHIVE_DRIVE, "-C", PREPARED_RESTORE_ROOT], check=True)
assert Path(CODEC_CHECKPOINT_DRIVE).exists(), f"Missing codec checkpoint: {CODEC_CHECKPOINT_DRIVE}"
shutil.copy2(CODEC_CHECKPOINT_DRIVE, CODEC_CHECKPOINT_LOCAL)
print("Prepared dataset:", PREPARED_DATA_DIR)
print("Codec checkpoint:", CODEC_CHECKPOINT_LOCAL)

In [ ]:
# Build the same fixed 2000-sample subset used for comparison runs
from pathlib import Path
import json, os, random

prepared_root = Path(PREPARED_DATA_DIR)
subset_root = Path(SUBSET_ROOT)
subset_data = Path(SUBSET_DATA_DIR)
subset_root.mkdir(parents=True, exist_ok=True)
subset_data.mkdir(parents=True, exist_ok=True)

wav_files = sorted(prepared_root.rglob("*.wav"))
assert len(wav_files) >= SUBSET_SIZE, f"Need at least {SUBSET_SIZE} wavs, found {len(wav_files)}"
selected = sorted(random.Random(SUBSET_SEED).sample(wav_files, SUBSET_SIZE))
train_count = int(SUBSET_SIZE * TRAIN_SPLIT)
train_files = selected[:train_count]
val_files = selected[train_count:]
train_set = set(train_files)

for src in selected:
    rel = src.relative_to(prepared_root)
    dst = subset_data / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    os.symlink(src, dst)
    meta_src = src.with_suffix('.am1.json')
    if meta_src.exists():
        meta_dst = dst.with_suffix('.am1.json')
        if meta_dst.exists() or meta_dst.is_symlink():
            meta_dst.unlink()
        os.symlink(meta_src, meta_dst)

with open(TRAIN_LIST_PATH, 'w', encoding='utf-8') as fp:
    for path in train_files:
        fp.write(str(path) + '\n')
with open(VAL_LIST_PATH, 'w', encoding='utf-8') as fp:
    for path in val_files:
        fp.write(str(path) + '\n')
with open(SUBSET_MANIFEST_PATH, 'w', encoding='utf-8') as fp:
    for path in selected:
        fp.write(json.dumps({'wav': str(path), 'split': 'train' if path in train_set else 'val'}) + '\n')

print('Subset built:', subset_root)
print('train:', len(train_files), 'val:', len(val_files))

In [ ]:
# Write runtime config
from pathlib import Path
import yaml

Path(RUNTIME_CONFIG_DIR).mkdir(parents=True, exist_ok=True)

cfg = {
    'seed': SUBSET_SEED,
    'model_type': 'codec',
    'data': {
        'sr': SR,
        'T': T,
        'scale': 1.0,
        'segment_mode': 'hapticgen',
        'normalize_mode': 'none',
        'min_segment_ratio': 1.0,
        'clip_range': None,
        'train_split': TRAIN_SPLIT,
        'analysis_batch_size': ANALYSIS_BATCH_SIZE,
        'train_random_seek': True,
        'train_sample_with_replacement': False,
        'val_random_seek': False,
        'val_sample_with_replacement': False,
        'train_file_list': TRAIN_LIST_PATH,
        'val_file_list': VAL_LIST_PATH,
    },
    'model': {
        'channels': CHANNELS,
        'strides': STRIDES,
        'first_kernel': FIRST_KERNEL,
        'kernel_size': KERNEL_SIZE,
        'residual_kernel_size': RESIDUAL_KERNEL_SIZE,
        'activation': 'leaky_relu',
        'norm': 'group',
        'code_dim': CODE_DIM,
        'n_codebooks': N_CODEBOOKS,
        'codebook_size': CODEBOOK_SIZE,
        'commitment_weight': COMMITMENT_WEIGHT,
        'control_dim': CONTROL_DIM,
        'control_hidden': CONTROL_HIDDEN,
        'metric_dim': 8,
    },
}
with open(BASELINE_CONFIG_PATH, 'w', encoding='utf-8') as fp:
    yaml.safe_dump(cfg, fp, sort_keys=False)
print(BASELINE_CONFIG_PATH)

In [ ]:
# Extract pooled-feature baseline and fit PCA + control-to-sequence regressor
%cd "$REPO_DIR"
BASELINE_DIR = f"{OUTPUT_RUN_DIR}/baseline"
!python scripts/extract_baseline_controls.py --config "$BASELINE_CONFIG_PATH" --data_dir "$PREPARED_DATA_DIR" --checkpoint "$CODEC_CHECKPOINT_LOCAL" --output_dir "$BASELINE_DIR" --n_components $N_COMPONENTS --ridge_alpha $RIDGE_ALPHA

In [ ]:
# Build controls outputs
CONTROLS_DIR = f"{OUTPUT_RUN_DIR}/controls"
!python scripts/build_controls_baseline.py --config "$BASELINE_CONFIG_PATH" --checkpoint "$CODEC_CHECKPOINT_LOCAL" --baseline_dir "$BASELINE_DIR" --output_dir "$CONTROLS_DIR"

In [ ]:
# Extended validation
VALIDATION_DIR = f"{OUTPUT_RUN_DIR}/validation"
!python scripts/validate_baseline.py --config "$BASELINE_CONFIG_PATH" --checkpoint "$CODEC_CHECKPOINT_LOCAL" --baseline_dir "$BASELINE_DIR" --output_dir "$VALIDATION_DIR" --controls_dir "$CONTROLS_DIR"

In [ ]:
# Save outputs back to Drive
from pathlib import Path
import json, shutil

export_root = Path(EXPORT_DIR)
export_root.mkdir(parents=True, exist_ok=True)
for src in [OUTPUT_RUN_DIR, BASELINE_CONFIG_PATH, TRAIN_LIST_PATH, VAL_LIST_PATH, SUBSET_MANIFEST_PATH]:
    src_path = Path(src)
    dst_path = export_root / src_path.name
    if src_path.is_dir():
        shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
    else:
        shutil.copy2(src_path, dst_path)

summary = {
    'run_name': RUN_NAME,
    'codec_checkpoint_local': CODEC_CHECKPOINT_LOCAL,
    'baseline_dir': BASELINE_DIR,
    'prepared_data_dir': PREPARED_DATA_DIR,
    'subset_root': SUBSET_ROOT,
    'export_dir': str(export_root),
}
with open(export_root / 'run_summary.json', 'w', encoding='utf-8') as fp:
    json.dump(summary, fp, indent=2)
print('Saved artifacts to:', export_root)